In [ ]:
!pip -q install -U tifffile imagecodecs pillow
import tifffile, imagecodecs, PIL

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import tensorflow as tf
import tifffile as tiff

tf.random.set_seed(42)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 112.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.49.1 requires pillow<12.0,>=8.0, but you have pillow 12.0.0 which is incompatible.
Mounted at /content/drive


In [ ]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")


# 1) 경로 설정


In [ ]:

DATA_ROOT = Path("/content/drive/MyDrive/mission3_data")


# --- Landsat ---
LANDSAT_IMG_TRAIN = DATA_ROOT/"U-Net_Dataset/train/TS_LS30_LS30_images"
LANDSAT_MASK_TRAIN = DATA_ROOT/"U-Net_Dataset/train/TL_LS30_labels"
LANDSAT_IMG_VAL   = DATA_ROOT/"U-Net_Dataset/valid/VS_LS30_LS30_images"
LANDSAT_MASK_VAL  = DATA_ROOT/"U-Net_Dataset/valid/VL_LS30_labels"

# --- Sentinel ---
SENT_IMG_TRAIN = DATA_ROOT/"U-Net_Dataset/train/images"
SENT_MASK_TRAIN = DATA_ROOT/"U-Net_Dataset/train/labels"
SENT_IMG_VAL   = DATA_ROOT/"U-Net_Dataset/valid/images"
SENT_MASK_VAL  = DATA_ROOT/"U-Net_Dataset/valid/labels"



# 2) 하이퍼파라미터


In [ ]:
IMG_SIZE = (512, 512)
CLIP_MAX = 10000.0
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
NUM_CLASSES = 1  # sigmoid(1채널)


# 3) 파일 매칭 함수


In [ ]:

def list_pairs(img_dir, lbl_dir):
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)
    imgs = sorted([p for p in img_dir.glob("*.tif")])
    lbls = []
    for p in imgs:
        q = lbl_dir / (p.stem + ".tif")
        if not q.exists():
            raise FileNotFoundError(f"라벨 없음: {q.name}")
        lbls.append(q)
    return imgs, lbls

ls_tr_imgs, ls_tr_lbls = list_pairs(LANDSAT_IMG_TRAIN, LANDSAT_MASK_TRAIN)
ls_va_imgs, ls_va_lbls = list_pairs(LANDSAT_IMG_VAL, LANDSAT_MASK_VAL)

st_tr_imgs, st_tr_lbls = list_pairs(SENT_IMG_TRAIN,   SENT_MASK_TRAIN)
st_va_imgs, st_va_lbls = list_pairs(SENT_IMG_VAL,     SENT_MASK_VAL)

print("Landsat train/val:", len(ls_tr_imgs), len(ls_va_imgs))
print("Sentinel train/val:", len(st_tr_imgs), len(st_va_imgs))

Landsat train/val: 4000 500
Sentinel train/val: 8000 1000





# 4) TIFF 읽기 + 마스크 처리


In [ ]:

def read_tif_multiband_float(path_str):
    arr = tiff.imread(path_str.numpy().decode()).astype(np.float32)
    if arr.ndim == 2:
        # 1채널이면 3채널로 복제
        arr = np.stack([arr, arr, arr], -1)
    # 값 클리핑 + 정규화
    arr = np.clip(arr, 0, CLIP_MAX) / CLIP_MAX
    return arr

def read_tif_mask_to_01(path_str):
    """TIFF 라벨을 {0,1} 이진 마스크로 변환: 10->1, 90->0 등"""
    path = path_str.numpy().decode()
    m = tiff.imread(path)
    if m.ndim == 3:
        m = m[..., 0]
    vals = np.unique(m)
    vset = set(vals.tolist())
    if vset == {10, 90}:
        m = (m == 10).astype(np.int32)
    elif vset == {0, 255}:
        m = (m > 0).astype(np.int32)
    elif vset.issubset({0, 1}):
        m = m.astype(np.int32)
    else:
        # 가장 큰 값을 positive로
        m = (m == vals.max()).astype(np.int32)
    return m

def load_pair(img_path, lbl_path):
    img = tf.py_function(read_tif_multiband_float, [img_path], tf.float32)
    msk = tf.py_function(read_tif_mask_to_01,      [lbl_path], tf.int32)

    # 동적 shape → 이후 resize에서 고정
    img.set_shape([None, None, None])
    msk.set_shape([None, None])

    # 리사이즈
    img = tf.image.resize(img, IMG_SIZE, method="bilinear")
    msk = tf.image.resize(msk[..., None], IMG_SIZE, method="nearest")  # (H,W,1)

    # float32 0/1 마스크로 변환
    msk = tf.cast(tf.cast(msk, tf.float32) > 0.5, tf.float32)
    return img, msk

def augment(img, m):
    if tf.random.uniform([]) < 0.5:
        img = tf.image.flip_left_right(img); m = tf.image.flip_left_right(m)
    if tf.random.uniform([]) < 0.5:
        img = tf.image.flip_up_down(img);   m = tf.image.flip_up_down(m)
    if tf.random.uniform([]) < 0.2:
        img = tf.image.random_brightness(img, 0.05)
    if tf.random.uniform([]) < 0.2:
        img = tf.image.random_contrast(img, 0.95, 1.05)
    return img, m

def make_ds(imgs, lbls, train=True, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(
        ([str(p) for p in imgs], [str(p) for p in lbls])
    )
    ds = ds.map(load_pair, num_parallel_calls=AUTOTUNE)
    if train:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(1024, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds

landsat_train_ds = make_ds(ls_tr_imgs, ls_tr_lbls, train=True)
landsat_valid_ds = make_ds(ls_va_imgs, ls_va_lbls, train=False)

sent_train_ds    = make_ds(st_tr_imgs, st_tr_lbls, train=True)
sent_valid_ds    = make_ds(st_va_imgs, st_va_lbls, train=False)


# U-Net 모델 정의

In [ ]:
from tensorflow.keras import layers, models

def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding="same")(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv2D(f, 3, padding="same")(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    return x

def enc(x, f):
    c = conv_block(x, f)
    p = layers.MaxPool2D(2)(c)
    return c, p

def dec(x, skip, f):
    x = layers.Conv2DTranspose(f, 2, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip])
    return conv_block(x, f)

# 배치에서 채널 수를 읽어 input_shape 자동 설정 (Landsat 기준으로)
example_x, _ = next(iter(landsat_train_ds))
H, W, C = example_x.shape[1], example_x.shape[2], example_x.shape[3]
print("Input shape:", H, W, C)

inp = layers.Input((H, W, C))
s1, p1 = enc(inp, 64)
s2, p2 = enc(p1, 128)
s3, p3 = enc(p2, 256)
s4, p4 = enc(p3, 512)

b = conv_block(p4, 1024)

d1 = dec(b,  s4, 512)
d2 = dec(d1, s3, 256)
d3 = dec(d2, s2, 128)
d4 = dec(d3, s1, 64)

out = layers.Conv2D(1, 1, activation="sigmoid")(d4)

model = models.Model(inp, out)
model.summary()


Input shape: 512 512 4


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 512, 512,  │          0 │ -                 │
│ (InputLayer)        │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 512, 512,  │      2,368 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 512, 512,  │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 512, 512,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 512, 512,  │     36,928 │ re_lu[0][0]       │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512, 512,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 512, 512,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 256, 256,  │          0 │ re_lu_1[0][0]     │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 256, 256,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 256, 256,  │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 256, 256,  │    147,584 │ re_lu_2[0][0]     │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        512 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 256, 256,  │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 128, 128,  │          0 │ re_lu_3[0][0]     │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 128, 128,  │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │      1,024 │ conv2d_4[0][0]  

 Total params: 31,055,873 (118.47 MB)

 Trainable params: 31,044,097 (118.42 MB)

 Non-trainable params: 11,776 (46.00 KB)

# 메트릭 정의 + 컴파일

In [ ]:
@tf.function
def _binarize(x):
    return tf.where(x >= 0.5, 1.0, 0.0)

@tf.function
def binary_dice(y_true, y_pred, eps=tf.constant(1e-7, tf.float32)):
    y_true_b, y_pred_b = _binarize(y_true), _binarize(y_pred)
    inter = tf.reduce_sum(y_true_b * y_pred_b)
    return (2.0 * inter + eps) / (tf.reduce_sum(y_true_b) + tf.reduce_sum(y_pred_b) + eps)

@tf.function
def binary_accuracy(y_true, y_pred):
    y_true_b, y_pred_b = _binarize(y_true), _binarize(y_pred)
    return tf.reduce_mean(tf.cast(tf.equal(y_true_b, y_pred_b), tf.float32))

# Keras BinaryIoU 사용
binary_iou_metric = tf.keras.metrics.BinaryIoU(threshold=0.5, name="mIoU")

# ==== 1단계: Landsat 사전학습용 컴파일 ====
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=[binary_iou_metric, binary_dice, binary_accuracy],
)




# Landsat 사전학습

In [ ]:
from pathlib import Path

CKPT_DIR = DATA_ROOT / "checkpoints_1125"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

CKPT_LS   = CKPT_DIR/"unet_pretrain_landsat.keras"     # 사전학습 가중치
CKPT_SENT = CKPT_DIR/"unet_sigmoid_best.keras"         # Sentinel 파인튜닝 최종

cbs_pretrain = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_mIoU", mode="max",
        factor=0.5, patience=5, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_mIoU", mode="max",
        patience=10, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        CKPT_LS, monitor="val_mIoU", mode="max",
        save_best_only=True, verbose=1
    ),
]

EPOCHS_PRETRAIN = 100  # 필요에 따라 조정

history_ls = model.fit(
    landsat_train_ds,
    validation_data=landsat_valid_ds,
    epochs=EPOCHS_PRETRAIN,
    callbacks=cbs_pretrain,
)

print("===== Landsat pretrain evaluation =====")
res_ls = model.evaluate(landsat_valid_ds, return_dict=True)
print(res_ls)


Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - binary_accuracy: 0.7303 - binary_dice: 0.6776 - loss: 0.5077 - mIoU: 0.5698
Epoch 1: val_mIoU improved from -inf to 0.31514, saving model to /content/drive/MyDrive/mission3_data/checkpoints_1125/unet_pretrain_landsat.keras
125/125 ━━━━━━━━━━━━━━━━━━━━ 934s 3s/step - binary_accuracy: 0.7306 - binary_dice: 0.6779 - loss: 0.5073 - mIoU: 0.5701 - val_binary_accuracy: 0.6316 - val_binary_dice: 3.5260e-14 - val_loss: 1.7768 - val_mIoU: 0.3151 - learning_rate: 3.0000e-04
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 507ms/step - binary_accuracy: 0.7692 - binary_dice: 0.7152 - loss: 0.4575 - mIoU: 0.6162
Epoch 2: val_mIoU did not improve from 0.31514
125/125 ━━━━━━━━━━━━━━━━━━━━ 75s 539ms/step - binary_accuracy: 0.7693 - binary_dice: 0.7154 - loss: 0.4572 - mIoU: 0.6164 - val_binary_accuracy: 0.6316 - val_binary_dice: 3.5260e-14 - val_loss: 3.1897 - val_mIoU: 0.3151 - learning_rate: 3.0000e-04
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 409ms

In [ ]:
# === Sentinel 파인튜닝: 더 낮은 lr로 재-컴파일 ===
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=[binary_iou_metric, binary_dice, binary_accuracy],
)

cbs_finetune = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_mIoU", mode="max",
        factor=0.5, patience=5, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_mIoU", mode="max",
        patience=12, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        CKPT_SENT, monitor="val_mIoU", mode="max",
        save_best_only=True, verbose=1
    ),
]

EPOCHS_FT = 100  # Sentinel 파인튜닝 epoch 수

history_ft = model.fit(
    sent_train_ds,
    validation_data=sent_valid_ds,
    epochs=EPOCHS_FT,
    callbacks=cbs_finetune,
)

print("===== Sentinel fine-tune evaluation =====")
res_sent = model.evaluate(sent_valid_ds, return_dict=True)
print(res_sent)  # 여기서 'mIoU' 확인


Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - binary_accuracy: 0.9081 - binary_dice: 0.7182 - loss: 0.2351 - mIoU: 0.7537
Epoch 1: val_mIoU improved from -inf to 0.74720, saving model to /content/drive/MyDrive/mission3_data/checkpoints_1125/unet_sigmoid_best.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 1270s 4s/step - binary_accuracy: 0.9082 - binary_dice: 0.7185 - loss: 0.2349 - mIoU: 0.7538 - val_binary_accuracy: 0.9239 - val_binary_dice: 0.7270 - val_loss: 0.2027 - val_mIoU: 0.7472 - learning_rate: 1.0000e-04
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 588ms/step - binary_accuracy: 0.9434 - binary_dice: 0.8226 - loss: 0.1403 - mIoU: 0.8181
Epoch 2: val_mIoU improved from 0.74720 to 0.76378, saving model to /content/drive/MyDrive/mission3_data/checkpoints_1125/unet_sigmoid_best.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 175s 650ms/step - binary_accuracy: 0.9434 - binary_dice: 0.8227 - loss: 0.1402 - mIoU: 0.8182 - val_binary_accuracy: 0.9291 - val_binary_dice: 0.7493 - val_loss: 0.1816 - val_m